## wls-var

MinTrace(method='wls_var') **order reconciliation** using the cleaned simulation pipeline.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

from inventory_pipeline import save_pickle, run_base_loop, run_fixed_loop

# order within training

In [2]:
gap1 = 365
gap2 = 1548
lead_time = 1

train = pd.read_pickle(f"{path1}721past_1913.pkl").reset_index(drop=True)[['ds', 'y']]
fit_all = pd.read_pickle(f"{path1}721fitts_base2cohe.pkl").reset_index(drop=True)
tr_all = pd.concat([train, fit_all], axis=1)

tr_tr = tr_all[tr_all['ds'] <= gap2].reset_index(drop=True).iloc[:, 1:]
tr_ts = tr_all[tr_all['ds'] > gap2].reset_index(drop=True).iloc[:, 1:]

MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
WLSVAR_NAMES = ['lgb_base', 'lgb_mint', 'ets_base', 'ets_mint']

In [3]:
# Training order histories for MinT(WLS-var)
trInvtSim = []
for name in MODEL_NAMES:
    df = run_base_loop(
        fcst=tr_ts['y'],
        truth=tr_ts['y'],
        residual=np.zeros(len(tr_tr['y'])),
        NAME=f"tr_{name}",
        gap1=gap1,
        gap2=gap2,
        L_=lead_time,
    )
    trInvtSim.append(df[['name', 'ot_90', 'ot_95', 'ot_99']])
actl365_L = pd.concat(trInvtSim, ignore_index=True)
actl365_L[['ot_90', 'ot_95', 'ot_99']] = actl365_L[['ot_90', 'ot_95', 'ot_99']].clip(lower=0)
save_pickle(actl365_L, f"365TRactual_L{lead_time}.pkl")

trInvtSim = []
for name in MODEL_NAMES:
    resid = (tr_tr[name] - tr_tr['y']).reset_index(drop=True)
    df = run_base_loop(
        fcst=tr_ts[name],
        truth=tr_ts['y'],
        residual=resid,
        NAME=f"tr_{name}",
        gap1=gap1,
        gap2=gap2,
        L_=lead_time,
    )
    trInvtSim.append(df[['name', 'ot_90', 'ot_95', 'ot_99']])
fcst365_L = pd.concat(trInvtSim, ignore_index=True)
fcst365_L[['ot_90', 'ot_95', 'ot_99']] = fcst365_L[['ot_90', 'ot_95', 'ot_99']].clip(lower=0)
save_pickle(fcst365_L, f"365TRfcst_L{lead_time}.pkl")

fcst365_L.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [06:18<00:00, 112.66it/s]


,name,ot_90,ot_95,ot_99
0,tr_lgb_base,5.638805,7.237328,10.235891
1,tr_lgb_base,0.000000,0.000000,0.000000
2,tr_lgb_base,0.000000,0.000000,0.000000
3,tr_lgb_base,3.637477,3.637477,3.637477
4,tr_lgb_base,2.446338,2.446338,2.446338


# WlsVar

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

from inventory_pipeline import save_pickle, run_base_loop, run_fixed_loop
gap1 = 365
gap2 = 1548
lead_time = 1

#fcst365_L = pd.read_pickle(f"365TRfcst_L{lead_time}.pkl")
#actl365_L = pd.read_pickle(f"365TRactual_L{lead_time}.pkl")
test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
S = pd.read_pickle(f"{path1}721S_df.pkl")
tags = pd.read_pickle(f"{path1}tags.bin")
ts_id = test[['unique_id', 'ds']].reset_index(drop=True)


In [2]:
train_full = pd.read_pickle(f"{path1}721past_1913.pkl").reset_index(drop=True)
tr_id = train_full[['unique_id', 'ds']].reset_index(drop=True)
trts_id = tr_id[tr_id['ds'] > gap2].reset_index(drop=True)

lgbs = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl").reset_index(drop=True)
etss = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl").reset_index(drop=True)
base_all = pd.concat([lgbs, etss], ignore_index=True)
MMint = base_all[['name', 'ot_90', 'ot_95', 'ot_99']].reset_index(drop=True)

tr_fo = pd.read_pickle(f"365TRfcst_L{lead_time}.pkl").reset_index(drop=True)
tr_do = pd.read_pickle(f"365TRactual_L{lead_time}.pkl").iloc[:, :2].reset_index(drop=True)
tr_do.columns = ['name', 'y']
tr365 = pd.concat([tr_do, tr_fo[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True)], axis=1)

mint = HierarchicalReconciliation(reconcilers=[MinTrace(method='wls_var')])
on = ['ot_90', 'ot_95', 'ot_99']

In [3]:
MINTo = []
MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
for name in tqdm(MODEL_NAMES):
    y_hat = pd.concat([ts_id, MMint[MMint['name'] == name][on].reset_index(drop=True)], axis=1)
    tr_order = pd.concat(
        [
            trts_id[['unique_id', 'ds']].reset_index(drop=True),
            tr365[tr365['name'] == f"tr_{name}"][['y', 'ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    cols_out = []
    for j in on:
        rec = mint.reconcile(
            Y_hat_df=y_hat[['unique_id', 'ds', j]],
            S=S,
            tags=tags,
            Y_df=tr_order[['unique_id', 'ds', 'y', j]],
        )
        cols_out.append(rec.iloc[:, -1].rename(j))

    rec_name = pd.concat(cols_out, axis=1).clip(lower=0)
    rec_name.insert(0, 'name', name)
    MINTo.append(rec_name)

OrderVar = pd.concat(MINTo, ignore_index=True)
save_pickle(OrderVar, f"OrderVar_L{lead_time}.pkl")
OrderVar.head()

100%|████████████████████████████████████████████████████████████████████████████████| 8/8 [3:23:45<00:00, 1528.18s/it]


,name,ot_90,ot_95,ot_99
0,lgb_base,10.467349,13.434492,18.999925
1,lgb_base,3.557998,3.557964,3.557878
2,lgb_base,4.807548,4.807558,4.807583
3,lgb_base,7.931240,7.931244,7.931255
4,lgb_base,0.783484,0.783501,0.783545


In [4]:
truth = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)['y']

Invt_df = []
for name in MODEL_NAMES:
    base_ref = base_all.loc[base_all['name'] == name].reset_index(drop=True)
    order_ref = OrderVar.loc[OrderVar['name'] == name].reset_index(drop=True)

    fixed_orders = pd.concat(
        [
            base_ref[['forecasts', 'sst_90', 'sst_95', 'sst_99']].reset_index(drop=True),
            order_ref[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    df = run_fixed_loop(
        truth=truth,
        forecast_source=base_ref['forecasts'].reset_index(drop=True),
        fixed_orders=fixed_orders,
        NAME=name,
        L_=lead_time,
    )
    Invt_df.append(df)

VarOrder = pd.concat(Invt_df, ignore_index=True)
save_pickle(VarOrder, f"VarOrder_L{lead_time}.pkl")
VarOrder.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:30<00:00, 473.69it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,10.467349,10.467349,12.157143,1.689794,0.0,...,0.0,10.253148,0.000000,18.999925,18.999925,20.689719,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,10.467349,3.557998,3.557998,10.715141,7.157143,0.0,...,0.0,10.253148,18.999925,3.557878,3.557878,19.247597,15.689719,0.0,15.689719,0.0
2,lgb_base,7.0,4.798928,5.648312,3.557998,4.807548,4.807548,8.522689,3.715141,0.0,...,0.0,10.253148,3.557878,4.807583,4.807583,17.055180,12.247597,0.0,12.247597,0.0
3,lgb_base,1.0,5.980217,5.648312,4.807548,7.931240,7.931240,15.453929,7.522689,0.0,...,0.0,10.253148,4.807583,7.931255,7.931255,23.986435,16.055180,0.0,16.055180,0.0
4,lgb_base,9.0,5.048564,5.648312,7.931240,0.783484,0.783484,7.237413,6.453929,0.0,...,0.0,10.253148,7.931255,0.783545,0.783545,15.769981,14.986435,0.0,14.986435,0.0
